# Phase 2 — DOI Merge and Author Extraction

Matches retracted papers to OpenAlex records by DOI and extracts the author
list of each.

**Input:** `data/interim/phase01_classified.csv`, the OpenAlex snapshot

**Outputs**

| File | Contents |
|---|---|
| `data/interim/phase02_author_paper.csv` | one row per author per retracted paper |
| `data/interim/phase02_paper_level.csv` | one row per matched paper |
| `data/interim/phase02_unmatched_dois.csv` | DOIs with no OpenAlex record |

Author names are never used for matching. The DOI merge returns OpenAlex author
identifiers directly, avoiding the disambiguation errors that name matching
would introduce in a sample that is approximately 47% China.

Byline position for the within-paper analysis is recorded here. Later
extraction phases do not retain the `authorships` field, so an author's
position on their other papers is not available elsewhere in the pipeline.

The extraction window is 2015–2022. The analysis window narrows to 2015–2019 at
Phase 8, where the balanced-panel requirement binds.

In [1]:
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from snapshot import Snapshot, concat, strip_doi, strip_id

CLASSIFIED = "data/interim/phase01_classified.csv"
OUT_AUTHORS = "data/interim/phase02_author_paper.csv"
OUT_PAPERS = "data/interim/phase02_paper_level.csv"
OUT_UNMATCHED = "data/interim/phase02_unmatched_dois.csv"

STUDY_YEARS = (2015, 2022)

# Restrict the scan for testing. None runs the full snapshot.
LIMIT_FILES = None
SKIP_FILES = 0

pd.set_option("display.width", 200)
os.makedirs("data/interim", exist_ok=True)

snap = Snapshot()
d = snap.describe("works")
print(f"works: {d['files']:,} files, {d['bytes'] / 2**30:.0f} GiB, "
      f"{d['partitions']:,} partitions")

works: 2,446 files, 675 GiB, 482 partitions


## Selecting DOIs

Retractions only. Expressions of concern and corrections describe a milder
action than the event this study measures. Records without a usable DOI, and
those whose reason labels are purely procedural, are excluded.

Where several records share a DOI, the earliest retraction is retained: it is
the first public signal.

In [2]:
rw = pd.read_csv(CLASSIFIED, low_memory=False)
rw["RetractionDate"] = pd.to_datetime(rw.RetractionDate, errors="coerce",
                                      format="mixed")
rw["doi_clean"] = rw.OriginalPaperDOI.apply(strip_doi)

print(f"records      {len(rw):,}")
print(f"DOI present  {rw.OriginalPaperDOI.notna().sum():,}")
print(f"DOI usable   {rw.doi_clean.notna().sum():,}")

sub = rw[
    (rw.RetractionYear >= STUDY_YEARS[0]) &
    (rw.RetractionYear <= STUDY_YEARS[1]) &
    (rw.RetractionNature == "Retraction") &
    (rw.doi_clean.notna()) &
    (~rw.Category.isin(["PROCESS_ONLY"]))
].copy()

print(f"\nstudy subset, {STUDY_YEARS[0]}-{STUDY_YEARS[1]}: {len(sub):,}")
print(sub.Category.value_counts().to_string())

n_before = len(sub)
sub = sub.sort_values("RetractionDate").drop_duplicates("doi_clean",
                                                        keep="first")
print(f"\ndeduplicated by DOI: {n_before:,} -> {len(sub):,}")

dois = sub.doi_clean.tolist()

records      71,799
DOI present  69,011
DOI usable   65,589

study subset, 2015-2022: 18,083
Category
AUTHOR_MISCONDUCT       10342
EDITORIAL_COMPROMISE     3135
HONEST_ERROR             2740
UNCONFIRMED_CONCERNS     1535
UNCLASSIFIED              170
ETHICS_VIOLATION          161

deduplicated by DOI: 18,083 -> 18,072


## Scan

A single pass over the `works` entity, retaining records whose DOI appears in
the selected set.

In [3]:
COLS = ["id", "doi", "publication_year", "publication_date", "type",
        "is_retracted", "cited_by_count", "authorships", "primary_location"]

t0 = time.time()
res = snap.works_for_dois(dois, columns=COLS,
                          limit_files=LIMIT_FILES, skip_files=SKIP_FILES,
                          progress_every=200)
works_tbl = concat(res)
print(f"\nelapsed {(time.time() - t0) / 60:.1f} min")
print(f"works matched: {works_tbl.num_rows if works_tbl is not None else 0:,}")

  matching against 18,072 dois
scanning works: 2,446 files, 675.2 GiB on disk
  projecting 9 of 49 columns
  200/2,446 files | 0.0M rows | kept 1 | 18 MiB/s | eta 645m
  400/2,446 files | 46.7M rows | kept 204 | 195 MiB/s | eta 55m
  600/2,446 files | 81.3M rows | kept 927 | 185 MiB/s | eta 54m
  800/2,446 files | 119.0M rows | kept 1,754 | 182 MiB/s | eta 51m
  1,000/2,446 files | 155.3M rows | kept 2,532 | 178 MiB/s | eta 48m
  1,200/2,446 files | 192.2M rows | kept 3,283 | 174 MiB/s | eta 45m
  1,400/2,446 files | 230.6M rows | kept 4,104 | 173 MiB/s | eta 41m
  1,600/2,446 files | 271.4M rows | kept 5,665 | 166 MiB/s | eta 38m
  1,800/2,446 files | 326.2M rows | kept 7,465 | 164 MiB/s | eta 32m
  2,000/2,446 files | 380.3M rows | kept 8,002 | 163 MiB/s | eta 25m
  2,200/2,446 files | 434.0M rows | kept 13,829 | 154 MiB/s | eta 13m
  2,400/2,446 files | 493.9M rows | kept 17,751 | 146 MiB/s | eta 2m
  done: 510,372,821 rows scanned, 18,214 kept, 80.6 min

elapsed 80.6 min
works matc

## Match rate

Loss is examined by category. Non-random loss in the editorial-compromise
category would compromise the arm the study uses as its no-attributed-fault
comparison.

In [4]:
if works_tbl is None or not works_tbl.num_rows:
    raise SystemExit("no works matched")

matched_dois = {strip_doi(d) for d in works_tbl.column("doi").to_pylist()}
matched_dois.discard(None)

sub["matched"] = sub.doi_clean.isin(matched_dois)
print(f"matched {sub.matched.sum():,} / {len(sub):,}  "
      f"({sub.matched.mean():.1%})\n")

by_arm = sub.groupby("Category").matched.agg(["sum", "count", "mean"])
by_arm.columns = ["matched", "total", "rate"]
by_arm["rate"] = (by_arm["rate"] * 100).round(1)
print(by_arm.to_string())

if "Publisher" in sub.columns:
    print("\nten largest publishers")
    pub = (sub.groupby("Publisher").matched.agg(["sum", "count", "mean"])
             .sort_values("count", ascending=False).head(10))
    pub.columns = ["matched", "total", "rate"]
    pub["rate"] = (pub["rate"] * 100).round(1)
    print(pub.to_string())

cols = [c for c in ["Record ID", "Title", "Journal", "Publisher", "Category",
                    "RetractionYear", "OriginalPaperDOI"] if c in sub.columns]
sub[~sub.matched][cols].to_csv(OUT_UNMATCHED, index=False)
print(f"\n{OUT_UNMATCHED}: {int((~sub.matched).sum()):,} rows")

matched 17,991 / 18,072  (99.6%)

                      matched  total   rate
Category                                   
AUTHOR_MISCONDUCT       10270  10336   99.4
EDITORIAL_COMPROMISE     3129   3131   99.9
ETHICS_VIOLATION          161    161  100.0
HONEST_ERROR             2735   2739   99.9
UNCLASSIFIED              167    170   98.2
UNCONFIRMED_CONCERNS     1529   1535   99.6

ten largest publishers
                                    matched  total   rate
Publisher                                                
Springer                               2682   2682  100.0
Elsevier                               2525   2525  100.0
Wiley                                  1368   1371   99.8
IOP Publishing                         1072   1072  100.0
Springer - Nature Publishing Group      850    850  100.0
Taylor and Francis                      757    757  100.0
Hindawi                                 616    616  100.0
PLoS                                    543    544   99.8
SAGE Publi

## Agreement with the OpenAlex retraction flag

Every paper here is retracted according to Retraction Watch. The comparison
records how often OpenAlex's own flag agrees, which is why that flag alone was
not used to construct the sample.

In [5]:
flagged = int(pc.sum(pc.fill_null(works_tbl.column("is_retracted"),
                                  False)).as_py())
print(f"is_retracted set on {flagged:,} of {works_tbl.num_rows:,} "
      f"({flagged / works_tbl.num_rows:.1%})")

is_retracted set on 16,676 of 18,214 (91.6%)


## Author table

`authorships` is a list of structs, one per author. Flattening it yields one
row per author per paper.

Authors without an OpenAlex identifier are excluded: an author who cannot be
identified cannot be followed across publications, so there is no history to
construct.

In [6]:
def explode_authorships(tbl):
    """One row per author per work."""
    auth_col = tbl.column("authorships")
    if isinstance(auth_col, pa.ChunkedArray):
        auth_col = auth_col.combine_chunks()

    parent = pc.list_parent_indices(auth_col).to_pylist()
    flat = auth_col.values
    author = flat.field("author")

    # First affiliation only.
    insts = flat.field("institutions").to_pylist()

    out = pd.DataFrame({
        "author_id": [strip_id(x) for x in author.field("id").to_pylist()],
        "author_display_name": author.field("display_name").to_pylist(),
        "position": flat.field("author_position").to_pylist(),
        "is_corresponding": flat.field("is_corresponding").to_pylist(),
        "country": [(i[0].get("country_code") if i else None) for i in insts],
        "institution_id": [(strip_id(i[0].get("id")) if i else None)
                           for i in insts],
        "_row": parent,
    })
    return out[out.author_id.notna()]


def n_authors_of(tbl):
    col = tbl.column("authorships")
    if isinstance(col, pa.ChunkedArray):
        col = col.combine_chunks()
    return pc.list_value_length(col)


work_meta = pd.DataFrame({
    "openalex_id": [strip_id(x) for x in works_tbl.column("id").to_pylist()],
    "doi": [strip_doi(x) for x in works_tbl.column("doi").to_pylist()],
    "pub_year": works_tbl.column("publication_year").to_pylist(),
    "n_authors": n_authors_of(works_tbl).to_pylist(),
    "source_id": [strip_id(((p or {}).get("source") or {}).get("id"))
                  if p else None
                  for p in works_tbl.column("primary_location").to_pylist()],
})
work_meta["_row"] = np.arange(len(work_meta))

ap = (explode_authorships(works_tbl)
      .merge(work_meta, on="_row", how="left")
      .drop(columns=["_row"]))

print(f"author-paper rows {len(ap):,}")
print(f"unique authors    {ap.author_id.nunique():,}")

author-paper rows 78,889
unique authors    63,758


In [7]:
meta = sub.set_index("doi_clean")
for src, dst in [("RetractionDate", "retraction_date"),
                 ("RetractionYear", "retraction_year"),
                 ("Category", "category"),
                 ("Journal", "rw_journal"),
                 ("Publisher", "rw_publisher")]:
    if src in meta.columns:
        ap[dst] = ap.doi.map(meta[src])

ORDER = ["author_id", "author_display_name", "position", "is_corresponding",
         "country", "institution_id", "doi", "openalex_id", "pub_year",
         "n_authors", "source_id", "retraction_date", "retraction_year",
         "category", "rw_journal", "rw_publisher"]
ap = ap[[c for c in ORDER if c in ap.columns]]

ap.to_csv(OUT_AUTHORS, index=False)
print(f"{OUT_AUTHORS}: {len(ap):,} rows, {len(ap.columns)} columns")

papers = ap.drop_duplicates("openalex_id")[
    ["openalex_id", "doi", "pub_year", "n_authors", "source_id",
     "retraction_date", "retraction_year", "category",
     "rw_journal", "rw_publisher"]]
papers.to_csv(OUT_PAPERS, index=False)
print(f"{OUT_PAPERS}: {len(papers):,} rows")

data/interim/phase02_author_paper.csv: 78,889 rows, 16 columns
data/interim/phase02_paper_level.csv: 17,591 rows


## Attrition into the author table

Two losses stand between a matched work and a row in the author table: works
for which OpenAlex holds no author list, and works whose author entries carry
no OpenAlex identifier.

In [8]:
n_works = works_tbl.num_rows
no_list = int(pc.sum(pc.equal(n_authors_of(works_tbl), 0)).as_py() or 0)
kept = ap.openalex_id.nunique()

print(f"works matched              {n_works:,}")
print(f"  no author list           {no_list:,}")
print(f"  authors without an id    {n_works - no_list - kept:,}")
print(f"papers in the author table {kept:,}")

att = pd.DataFrame({
    "matched": sub[sub.matched].groupby("Category").size(),
    "in_table": papers.groupby("category").size(),
}).fillna(0).astype(int)
att["rate"] = (100 * att.in_table / att.matched).round(1)
print(f"\nretention by category")
print(att.to_string())
print(f"\nspread across categories: "
      f"{att['rate'].max() - att['rate'].min():.1f} points")

no_id_years = None
if n_works - no_list - kept > 0:
    got = set(ap.openalex_id)
    ids = [strip_id(x) for x in works_tbl.column("id").to_pylist()]
    yrs = works_tbl.column("publication_year").to_pylist()
    lens = n_authors_of(works_tbl).to_pylist()
    no_id_years = pd.Series([y for i, y, n in zip(ids, yrs, lens)
                             if i not in got and n], name="pub_year")
    print(f"\npublication year of works whose authors carry no identifier")
    print(no_id_years.value_counts().sort_index().tail(12).to_string())

works matched              18,214
  no author list           296
  authors without an id    327
papers in the author table 17,591

retention by category
                      matched  in_table  rate
AUTHOR_MISCONDUCT       10270     10047  97.8
EDITORIAL_COMPROMISE     3129      3056  97.7
ETHICS_VIOLATION          161       160  99.4
HONEST_ERROR             2735      2696  98.6
UNCLASSIFIED              167       162  97.0
UNCONFIRMED_CONCERNS     1529      1470  96.1

spread across categories: 3.3 points

publication year of works whose authors carry no identifier
pub_year
2011     3
2012     7
2013     6
2014     9
2015    10
2016    20
2017    17
2018    42
2019    96
2020    46
2021    52
2022    15


## Authors and byline positions

The within-paper analysis compares lead authors, first or last, against middle
authors on the same paper. Every cell of the position-by-category table must be
populated for that comparison to be estimable within each category.

Middle authorship exists only on papers with three or more authors, so the
share of shorter papers bounds what that design can be estimated on.

In [9]:
print(f"unique authors    {ap.author_id.nunique():,}")
print(f"author-paper rows {len(ap):,}")
print(f"papers            {ap.openalex_id.nunique():,}")

print(f"\nbyline position")
print(ap.position.value_counts(dropna=False).to_string())

print(f"\nposition by category")
print(pd.crosstab(ap.position, ap.category).to_string())

print(f"\ncorresponding-author flag present: "
      f"{ap.is_corresponding.notna().mean():.1%}")
print(f"country present on affiliation:    {ap.country.notna().mean():.1%}")

unique authors    63,758
author-paper rows 78,889
papers            17,591

byline position
position
middle    46823
first     17374
last      14692

position by category
category  AUTHOR_MISCONDUCT  EDITORIAL_COMPROMISE  ETHICS_VIOLATION  HONEST_ERROR  UNCLASSIFIED  UNCONFIRMED_CONCERNS
position                                                                                                             
first                  9925                  3007               157          2672           161                  1452
last                   8307                  2424               149          2467           128                  1217
middle                26504                  5730               799          9651           312                  3827

corresponding-author flag present: 100.0%
country present on affiliation:    90.0%


In [10]:
n_auth = ap.drop_duplicates("openalex_id").n_authors

print(f"authors per retracted paper")
print(f"  median  {n_auth.median():.0f}")
print(f"  mean    {n_auth.mean():.1f}")
print(f"  range   {int(n_auth.min())} to {int(n_auth.max())}")
print()
for lo, hi, label in [(1, 1, "single author"), (2, 2, "two authors"),
                      (3, 5, "three to five"), (6, 10, "six to ten"),
                      (11, 10**6, "more than ten")]:
    n = int(((n_auth >= lo) & (n_auth <= hi)).sum())
    print(f"  {label:<16}{n:>7,}  ({n / len(n_auth):>5.1%})")

no_middle = int((n_auth < 3).sum())
print(f"\npapers with no middle author: {no_middle:,} "
      f"({no_middle / len(n_auth):.1%})")

per = ap.groupby("author_id").openalex_id.nunique()
multi = ap[ap.author_id.isin(per[per > 1].index)]
spans = multi.groupby("author_id").category.nunique()
print(f"\nretracted papers per author")
print(per.value_counts().sort_index().head(6).to_string())
print(f"\n  more than one:              {int((per > 1).sum()):,} "
      f"({(per > 1).mean():.1%})")
print(f"  spanning several categories: {int((spans > 1).sum()):,} "
      f"({(spans > 1).mean():.1%} of those)")

authors per retracted paper
  median  4
  mean    4.7
  range   1 to 690

  single author     2,581  (14.7%)
  two authors       2,741  (15.6%)
  three to five     6,986  (39.7%)
  six to ten        4,409  (25.1%)
  more than ten       874  ( 5.0%)

papers with no middle author: 5,322 (30.3%)

retracted papers per author
openalex_id
1    56417
2     4890
3     1192
4      483
5      283
6      140

  more than one:              7,341 (11.5%)
  spanning several categories: 2,250 (30.6% of those)


## Composition

Country of first affiliation bears on the author disambiguation limitation:
OpenAlex entity errors are more frequent for common and non-Western names. The
lag from publication to retraction bears on the interpretation of the event as
the end of a process rather than its beginning.

In [11]:
print("country of first affiliation, ten most frequent")
for c, n in ap.country.value_counts(dropna=False).head(10).items():
    print(f"  {str(c):<8}{n:>8,}  ({n / len(ap):>5.1%})")
print(f"\naffiliation absent: {ap.country.isna().mean():.1%}")

print(f"\nretraction year")
print(sub[sub.matched].RetractionYear.value_counts().sort_index().to_string())

lag = sub[sub.matched].LagYears.dropna()
print(f"\nyears from publication to retraction")
print(f"  median          {lag.median():.1f}")
print(f"  mean            {lag.mean():.1f}")
print(f"  within 2 years  {(lag <= 2).mean():.1%}")

country of first affiliation, ten most frequent
  CN        30,875  (39.1%)
  US         9,418  (11.9%)
  None       7,907  (10.0%)
  IN         5,229  ( 6.6%)
  JP         2,694  ( 3.4%)
  IR         2,368  ( 3.0%)
  IT         1,595  ( 2.0%)
  KR         1,383  ( 1.8%)
  GB         1,252  ( 1.6%)
  PK         1,201  ( 1.5%)

affiliation absent: 10.0%

retraction year
RetractionYear
2015.0    1228
2016.0    1288
2017.0    1228
2018.0    1435
2019.0    1556
2020.0    2368
2021.0    3501
2022.0    5387

years from publication to retraction
  median          1.7
  mean            3.0
  within 2 years  56.3%


## Input to Phase 3

In [12]:
ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]

print(f"authors to screen                {ap.author_id.nunique():,}")
print(f"  in the three study categories  "
      f"{ap[ap.category.isin(ARMS)].author_id.nunique():,}")

pos = pd.crosstab(ap[ap.category.isin(ARMS)].position,
                  ap[ap.category.isin(ARMS)].category)
print(f"\nposition by category, study categories only")
print(pos.to_string())
print(f"\nsmallest cell: {pos.values.min():,}")

authors to screen                63,758
  in the three study categories  57,059

position by category, study categories only
category  AUTHOR_MISCONDUCT  EDITORIAL_COMPROMISE  HONEST_ERROR
position                                                       
first                  9925                  3007          2672
last                   8307                  2424          2467
middle                26504                  5730          9651

smallest cell: 2,424
